# 06 - Deep baseline (PrositLike, trained properly)

Same `PrositLike` architecture and masked spectral-angle loss as notebook 05, unchanged. What differs:

* fixed leak-free split from `make_split.py` (`splits.parquet`: grouped by stripped sequence, stratified by method x charge, z in {2,3,4})
* full train split (~284k spectra), early stopping on the val split (patience 5, max 50 epochs), best checkpoint kept
* results appended to `results/runs.jsonl`

**The test split is never loaded here.**

All logic lives in `train_deep.py`; this notebook only sets paths and calls it. Works locally and on Colab (GPU: Runtime -> Change runtime type -> T4/L4/A100).

## Config: edit paths here

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "day1-baseline"
REPO_URL = "https://github.com/LordBurtz/sommer.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/sommer"          # folder containing spectra_with_charge.parquet
    SAVE_DIR = "/content/drive/MyDrive/sommer/day1"     # results/checkpoints get copied here at the end
    if not os.path.exists("/content/sommer"):
        !git clone -b {BRANCH} {REPO_URL} /content/sommer
    os.chdir("/content/sommer")
else:
    DATA_DIR = "."
    SAVE_DIR = None

SPECTRA_FILE = "spectra_with_charge.parquet"            # resolved against DATA_DIR
SPLIT_FILE = os.path.abspath("splits.parquet")          # committed in the repo
OUT_DIR = "."                                           # cache/, checkpoints/, results/ go here
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| data:", DATA_DIR)

## Smoke test (~2k spectra, 1 epoch)
Writes to `results/smoke_runs.jsonl`, not to `runs.jsonl`. The first call also encodes and caches the train+val arrays (~30 s).

In [ ]:
import train_deep

common_args = ["--data-dir", DATA_DIR, "--spectra-file", SPECTRA_FILE,
               "--split-file", SPLIT_FILE, "--out-dir", OUT_DIR]
_ = train_deep.main(common_args + ["--smoke"])

## Full training run
~10 min/epoch on a 4-core CPU; expect a few minutes per epoch or less on a Colab GPU.

In [ ]:
record = train_deep.main(common_args)

## Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

runs = [json.loads(l) for l in open(os.path.join(OUT_DIR, "results/runs.jsonl"))]
print(pd.DataFrame([{
    "timestamp": r["timestamp"], "commit": r["git_commit"][:8], "device": r["device"],
    "n_train": r["n_train"], "epochs": r["epochs_trained"], "best_epoch": r["best_epoch"],
    "val_SA": r["val_spectral_angle"], "val_pearson": r["val_pearson"],
    **{f"SA_z{z}": v["spectral_angle"] for z, v in r["by_charge"].items()},
} for r in runs]).round(4).to_string(index=False))
display(Image(os.path.join(OUT_DIR, "results/deep_baseline_curve.png")))

## Colab: persist outputs to Drive
The Colab clone is discarded with the runtime. Copy `results/runs.jsonl` back into the repo and commit it.

In [ ]:
if SAVE_DIR:
    import shutil
    os.makedirs(SAVE_DIR, exist_ok=True)
    for d in ("results", "checkpoints"):
        shutil.copytree(os.path.join(OUT_DIR, d), os.path.join(SAVE_DIR, d), dirs_exist_ok=True)
    print("copied results/ and checkpoints/ to", SAVE_DIR)